In [ ]:
# =============================================================
# CELL 1 - KONFIGURASI (satu-satunya cell yang boleh diedit)
# =============================================================
CFG = {
    # ---- Path ----
    "DATA_DIR": None,            # None = auto-discovery dari /kaggle/input atau ./dataset
    "WORK_DIR": "/kaggle/working",

    # ---- Mode eksekusi ----
    "RUN_MODE": "smoke",         # smoke | cv | final
    "MODEL": "graph_stid",       # graph_stid | graph_wavenet
    "FOLDS": [2],                # [2] untuk cepat, [0, 1, 2] untuk CV penuh

    # ---- Fitur ----
    "USE_GRAPH": True,
    "GRAPH_HOPS": 2,
    "USE_ROAD_METADATA": True,
    "USE_TEXT": False,

    # ---- Arsitektur ----
    "HIDDEN_DIM": 128,
    "ROAD_EMBED_DIM": 32,
    "TEXT_EMBED_DIM": 32,

    # ---- Training ----
    "BATCH_SIZE": 16,
    "EPOCHS": 40,
    "PATIENCE": 7,
    "LR": 1e-3,
    "WEIGHT_DECAY": 1e-4,
    "DROPOUT": 0.10,
    "GRAD_CLIP": 1.0,
    "AMP": True,
    "SEED": 42,
    "NUM_WORKERS": 2,

    # ---- Inferensi ----
    "PREDICTION_CLIP": [0.0, 160.0],
    "MAKE_SUBMISSION": False,
}

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">
<br>
<h1 style="font-family:verdana; font-size:26px"><center>~ Datathon 2026 - Task 1: Peramalan Kecepatan Lalu Lintas ~</center></h1>
<h3 style="font-family:verdana; font-size:14px; color:gray"><center>Residual Graph-STID + Mean-Reversion Baseline | Horizon h5, h10, h15</center></h3>
<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Daftar Isi

1. [**Konfigurasi**](#1)
2. [**Inisialisasi**](#2)
3. [**Penemuan Path dan Validasi File**](#3)
4. [**Muat Data**](#4)
5. [**Profil Data dan Deteksi Regime**](#5)
6. [**Pembangunan Fold dan Pemeriksaan Kebocoran**](#6)
7. [**Baseline Mean-Reversion**](#7)
8. [**Normalisasi dan Prakomputasi Fitur Graf**](#8)
9. [**Dataset dan DataLoader**](#9)
10. [**Residual Graph-STID**](#10)
11. [**Compact Graph WaveNet**](#11)
12. [**Mesin Pelatihan**](#12)
13. [**Evaluasi MSE Skala Asli**](#13)
14. [**Eksekusi Smoke / CV**](#14)
15. [**Ringkasan Ranking dan Acceptance Gate**](#15)
16. [**Pelatihan Final**](#16)
17. [**Inferensi Test**](#17)
18. [**Ensemble OOF per Horizon**](#18)
19. [**Pembuatan Submission**](#19)
20. [**Ringkasan Hasil**](#20)

---

## Tujuan

Notebook ini melatih, memvalidasi, membandingkan, dan menghasilkan *submission* model prakiraan kecepatan lalu lintas tanpa mengubah *cell* selain *cell* konfigurasi.

**Arsitektur prediksi:**

```
prediksi_akhir = prediksi_mean_reversion + residual_model
```

Model utama adalah *Residual Graph-STID* yang menggabungkan riwayat kecepatan, agregasi graf, identitas ruas, dan metadata jalan. *Compact Graph WaveNet* menjadi model penantang.

**Metrik resmi:** MSE pada skala kecepatan asli (km/jam).

In [ ]:
# =============================================================
# CELL 3 - ENVIRONMENT CHECK (nvidia-smi)
# =============================================================
import subprocess, sys
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout[:500])
else:
    print('nvidia-smi tidak tersedia atau tidak ada GPU.')
print('Python:', sys.version.split()[0])

In [ ]:
# =============================================================
# CELL 4 - IMPORTS, SEED, VERSI
# =============================================================
import json
import os
import platform
import random
import time
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import scipy.sparse as sp

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import ConcatDataset, DataLoader, Dataset
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("PERINGATAN: PyTorch tidak tersedia.")

warnings.filterwarnings("ignore")


def set_seed(seed: int) -> None:
    """Mengunci seluruh sumber keacakan agar hasil dapat direproduksi."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(CFG["SEED"])

print("Python  :", platform.python_version())
print("OS      :", platform.system(), platform.release())
print("numpy   :", np.__version__)
print("pandas  :", pd.__version__)

if TORCH_AVAILABLE:
    print("torch   :", torch.__version__)
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            p = torch.cuda.get_device_properties(i)
            print(f"GPU {i}   : {p.name} | VRAM {p.total_memory / 1e9:.1f} GB")
        DEVICE = torch.device("cuda")
    else:
        print("GPU     : tidak tersedia, memakai CPU")
        DEVICE = torch.device("cpu")
else:
    DEVICE = None

print("CWD     :", os.getcwd())
# --- Guard: stub kelas PyTorch untuk lingkungan tanpa GPU ---
if not TORCH_AVAILABLE:
    class _DummyModule:
        def __init__(self, *a, **kw): pass
        def __init_subclass__(cls, **kw): pass
    # Stub minimal agar definisi kelas tidak crash tanpa torch
    class nn:
        Module = _DummyModule
        class ModuleList(list): pass
        class ModuleDict(dict): pass
        class Embedding(_DummyModule): pass
        class Linear(_DummyModule): pass
        class LayerNorm(_DummyModule): pass
        class GELU(_DummyModule): pass
        class ReLU(_DummyModule): pass
        class Dropout(_DummyModule): pass
        class Sequential(_DummyModule): pass
        class Parameter(_DummyModule): pass
        class utils:
            @staticmethod
            def clip_grad_norm_(*a, **kw): pass
    import sys as _sys
    _sys.modules.setdefault('torch.nn', nn)
    class torch:
        class Tensor: pass
        @staticmethod
        def from_numpy(x): return x
        @staticmethod
        def zeros(*a, **kw): return None
        @staticmethod
        def arange(*a, **kw): return None
        @staticmethod
        def no_grad():
            class _ng:
                def __enter__(self): return self
                def __exit__(self, *a): pass
            return _ng()
        class optim:
            class AdamW:
                def __init__(self, *a, **kw): pass
            class lr_scheduler:
                class CosineAnnealingLR:
                    def __init__(self, *a, **kw): pass
        class cuda:
            @staticmethod
            def is_available(): return False
            @staticmethod
            def device_count(): return 0
        class amp:
            @staticmethod
            def GradScaler(*a, **kw): return None
            @staticmethod
            def autocast(*a, **kw):
                class _cm:
                    def __enter__(self): return self
                    def __exit__(self, *a): pass
                return _cm()
        class backends:
            class cudnn: pass
        device = str
        float32 = 'float32'
        bool = 'bool'
        nn = nn
    Dataset = object
    DataLoader = list
    ConcatDataset = list
    DEVICE = None


In [ ]:
# =============================================================
# CELL 5 - PENEMUAN PATH OTOMATIS DAN VALIDASI FILE
# =============================================================

def discover_data_root(cfg_data_dir: Optional[str]) -> Path:
    """Menemukan direktori data secara otomatis berdasarkan environment.

    Urutan prioritas:
    1. CFG["DATA_DIR"] jika tidak None
    2. Path Kaggle standar (rekursif)
    3. Folder dataset/ lokal
    """
    if cfg_data_dir is not None:
        root = Path(cfg_data_dir)
        if root.exists():
            return root
        raise FileNotFoundError(f"DATA_DIR yang dikonfigurasi tidak ditemukan: {root}")

    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates = list(kaggle_root.rglob("test_X_hist.npy"))
        if candidates:
            for c in candidates:
                parent = c.parent.parent
                if (parent / "train").exists() and (parent / "static").exists():
                    return parent
        raise FileNotFoundError(
            "Tidak menemukan struktur dataset di /kaggle/input. "
            "Pastikan dataset telah ditambahkan ke notebook."
        )

    local = Path("dataset")
    if local.exists():
        return local

    raise FileNotFoundError(
        "Tidak menemukan direktori data. Atur CFG['DATA_DIR'] secara eksplisit."
    )


DATA_ROOT = discover_data_root(CFG["DATA_DIR"])

# WORK_DIR: fallback ke subfolder lokal jika path Kaggle tidak dapat dibuat
_work_dir_candidate = Path(CFG["WORK_DIR"])
try:
    _work_dir_candidate.mkdir(parents=True, exist_ok=True)
    WORK_DIR = _work_dir_candidate
except (PermissionError, OSError):
    WORK_DIR = Path("outputs_local")
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    print(f"PERINGATAN: '{CFG['WORK_DIR']}' tidak dapat dibuat. Menggunakan '{WORK_DIR}' sebagai gantinya.")

(WORK_DIR / "outputs").mkdir(exist_ok=True)

PATHS = {
    "train_speed_m1": DATA_ROOT / "train" / "train_speed_m1_1_11160.npy",
    "train_speed_m2": DATA_ROOT / "train" / "train_speed_m2_1_5039.npy",
    "train_text_m1" : DATA_ROOT / "train" / "train_text_m1_1_11160.json",
    "train_text_m2" : DATA_ROOT / "train" / "train_text_m2_1_5039.json",
    "test_hist"     : DATA_ROOT / "test"  / "test_X_hist.npy",
    "test_texts"    : DATA_ROOT / "test"  / "test_texts.json",
    "adjacency"     : DATA_ROOT / "static" / "matrix.npy",
    "roads_meta"    : DATA_ROOT / "static" / "Roads1260.json",
    "active_mask"   : DATA_ROOT / "static" / "active_mask.npy",
    "sample_sub"    : DATA_ROOT / "sample_submission.csv",
}

print(f"Environment : {'Kaggle' if Path('/kaggle/input').exists() else 'Lokal'}")
print(f"DATA_ROOT   : {DATA_ROOT}")
print(f"WORK_DIR    : {WORK_DIR}")
print()
missing = []
for name, p in PATHS.items():
    status = "OK    " if p.exists() else "HILANG"
    print(f"  {name:18s} {status}  {p}")
    if not p.exists():
        missing.append(name)

if missing:
    raise FileNotFoundError(f"File tidak ditemukan: {missing}")
print("\nSeluruh file terverifikasi.")

In [ ]:
# =============================================================
# CELL 6 - MUAT DATA
# =============================================================

def _ordered_json(d: dict) -> list:
    return [d[k] for k in sorted(d, key=lambda s: int(s.split("_")[-1]))]


t0 = time.time()

speed_m1  = np.load(PATHS["train_speed_m1"])   # (11160, 1260)
speed_m2  = np.load(PATHS["train_speed_m2"])   # (5039, 1260)
test_hist = np.load(PATHS["test_hist"])         # (540, 15, 1260)

adj_raw    = np.load(PATHS["adjacency"]).astype(np.float32)  # (1260, 1260)
np.fill_diagonal(adj_raw, 0)
active_mask = np.load(PATHS["active_mask"])
roads_meta  = json.load(open(PATHS["roads_meta"]))
sample_sub  = pd.read_csv(PATHS["sample_sub"])

raw_text_m1 = json.load(open(PATHS["train_text_m1"]))
raw_text_m2 = json.load(open(PATHS["train_text_m2"]))
raw_text_te = json.load(open(PATHS["test_texts"]))
texts_m1 = _ordered_json(raw_text_m1)
texts_m2 = _ordered_json(raw_text_m2)
texts_te = _ordered_json(raw_text_te)
del raw_text_m1, raw_text_m2, raw_text_te

N_ROADS  = 1260
HIST_LEN = 15
HORIZONS = (5, 10, 15)
N_TEST   = 540

assert speed_m1.shape  == (11160, N_ROADS), f"Shape M1 salah: {speed_m1.shape}"
assert speed_m2.shape  == (5039,  N_ROADS), f"Shape M2 salah: {speed_m2.shape}"
assert test_hist.shape == (N_TEST, HIST_LEN, N_ROADS), f"Shape test salah: {test_hist.shape}"
assert adj_raw.shape   == (N_ROADS, N_ROADS), f"Shape matrix salah: {adj_raw.shape}"
assert np.isfinite(speed_m1).all(),  "Nilai non-finite pada speed_m1"
assert np.isfinite(speed_m2).all(),  "Nilai non-finite pada speed_m2"
assert np.isfinite(test_hist).all(), "Nilai non-finite pada test_hist"
assert len(texts_m1)  == len(speed_m1),  "Panjang teks M1 tidak selaras"
assert len(texts_m2)  == len(speed_m2),  "Panjang teks M2 tidak selaras"
assert len(texts_te)  == N_TEST,         "Panjang teks test tidak selaras"
assert len(roads_meta) == N_ROADS,       f"Jumlah segmen salah: {len(roads_meta)}"

print(f"Muat data selesai dalam {time.time() - t0:.1f} dtk")
for name, a in [("speed_m1", speed_m1), ("speed_m2", speed_m2), ("test_hist", test_hist)]:
    print(f"  {name:12s} shape={str(a.shape):20s} dtype={str(a.dtype):8s} "
          f"min={a.min():.2f} max={a.max():.2f}")
print(f"  adjacency    shape={str(adj_raw.shape):20s} tepi={int(adj_raw.sum())}")
print(f"  sample_sub   shape={str(sample_sub.shape)}")

In [ ]:
# =============================================================
# CELL 7 - PROFIL DATA DAN DETEKSI REGIME
# =============================================================

ZERO_FRAC_M1   = float((speed_m1  == 0).mean())
ZERO_FRAC_M2   = float((speed_m2  == 0).mean())
ZERO_FRAC_TEST = float((test_hist == 0).mean())

print("Profil data:")
print(f"  M1  : {speed_m1.shape[0]:5d} timestep, fraksi nol = {ZERO_FRAC_M1:.4f} ({ZERO_FRAC_M1:.1%})")
print(f"  M2  : {speed_m2.shape[0]:5d} timestep, fraksi nol = {ZERO_FRAC_M2:.4f} ({ZERO_FRAC_M2:.1%})")
print(f"  Test: {N_TEST:5d} sampel,   fraksi nol = {ZERO_FRAC_TEST:.4f} ({ZERO_FRAC_TEST:.1%})")


def infer_regimes(test_windows: np.ndarray, prototypes: dict) -> np.ndarray:
    """Menetapkan regime (m1 atau m2) untuk setiap sampel test berdasarkan fraksi nol.

    Args:
        test_windows: array shape (N, HIST, ROADS)
        prototypes: dict {"m1": float, "m2": float} fraksi nol prototipe

    Returns:
        array string "m1" atau "m2" untuk setiap sampel
    """
    zero_fracs = (test_windows == 0).mean(axis=(1, 2))
    threshold  = (prototypes["m1"] + prototypes["m2"]) / 2.0
    return np.where(zero_fracs < threshold, "m1", "m2")


PROTO_M1 = ZERO_FRAC_M1
PROTO_M2 = ZERO_FRAC_M2
prototypes  = {"m1": PROTO_M1, "m2": PROTO_M2}
test_regimes = infer_regimes(test_hist, prototypes)

N_M1_TEST = int((test_regimes == "m1").sum())
N_M2_TEST = int((test_regimes == "m2").sum())

print(f"\nKlasifikasi regime test:")
print(f"  M1: {N_M1_TEST} sampel")
print(f"  M2: {N_M2_TEST} sampel")

assert N_M1_TEST == 372, f"Jumlah sampel M1 test diharapkan 372, dapat {N_M1_TEST}"
assert N_M2_TEST == 168, f"Jumlah sampel M2 test diharapkan 168, dapat {N_M2_TEST}"
print("Assertsi regime test lulus: 372 M1 / 168 M2.")

always_zero_m1 = (speed_m1 == 0).all(axis=0)  # (1260,) bool
always_zero_m2 = (speed_m2 == 0).all(axis=0)  # (1260,) bool
print(f"\nRuas yang selalu nol: M1={always_zero_m1.sum()}, M2={always_zero_m2.sum()}")

REGIME_WEIGHTS = {"m1": N_M1_TEST / N_TEST, "m2": N_M2_TEST / N_TEST}
print(f"Bobot evaluasi: m1={REGIME_WEIGHTS['m1']:.4f}, m2={REGIME_WEIGHTS['m2']:.4f}")

In [ ]:
# =============================================================
# CELL 8 - PEMBANGUNAN FOLD DAN PEMERIKSAAN KEBOCORAN
# =============================================================

# Fold 0: train [0%,55%]  val [55%,70%]
# Fold 1: train [0%,70%]  val [70%,85%]
# Fold 2: train [0%,85%]  val [85%,100%]
FOLD_BOUNDS = [
    (0.55, 0.55, 0.70),
    (0.70, 0.70, 0.85),
    (0.85, 0.85, 1.00),
]


def make_windows(
    speed: np.ndarray,
    origins: np.ndarray,
    history: int = 15,
    horizons: tuple = (5, 10, 15),
) -> Tuple[np.ndarray, np.ndarray]:
    """Membentuk jendela riwayat dan target dari deret kecepatan.

    Args:
        speed  : (T, N_ROADS)
        origins: (W,) indeks timestep akhir riwayat
        history: panjang riwayat
        horizons: tuple horizon target

    Returns:
        histories: (W, history, N_ROADS)
        targets  : (W, len(horizons), N_ROADS)
    """
    idx_hist = origins[:, None] - np.arange(history - 1, -1, -1)[None, :]
    histories = speed[idx_hist]                                       # (W, H, N)
    targets   = np.stack([speed[origins + h] for h in horizons], axis=1)  # (W, 3, N)
    return histories, targets


def build_fold_origins(
    n_timesteps: int,
    train_end_frac: float,
    val_start_frac: float,
    val_end_frac: float,
    history: int = 15,
    max_horizon: int = 15,
) -> Tuple[np.ndarray, np.ndarray]:
    """Menghasilkan indeks origin train dan val untuk satu fold."""
    min_origin = history - 1
    max_origin = n_timesteps - max_horizon - 1

    train_end_idx  = int(n_timesteps * train_end_frac) - 1
    val_start_idx  = int(n_timesteps * val_start_frac) + history - 1
    val_end_idx    = int(n_timesteps * val_end_frac)   - max_horizon

    train_origins = np.arange(min_origin, min(train_end_idx, max_origin) + 1)
    val_origins   = np.arange(
        max(val_start_idx, min_origin),
        min(val_end_idx,   max_origin) + 1,
    )

    # Pemeriksaan kebocoran: riwayat validasi harus sepenuhnya setelah batas train
    if len(val_origins) > 0:
        earliest_val_hist = val_origins[0] - (history - 1)
        assert earliest_val_hist >= int(n_timesteps * train_end_frac), (
            f"KEBOCORAN: riwayat validasi ({earliest_val_hist}) tumpang tindih "
            f"dengan train (batas={int(n_timesteps * train_end_frac)})"
        )

    return train_origins, val_origins


FOLDS_M1, FOLDS_M2 = [], []
for i, (tr1, va0, va1) in enumerate(FOLD_BOUNDS):
    tr_m1, va_m1 = build_fold_origins(len(speed_m1), tr1, va0, va1)
    tr_m2, va_m2 = build_fold_origins(len(speed_m2), tr1, va0, va1)
    FOLDS_M1.append((tr_m1, va_m1))
    FOLDS_M2.append((tr_m2, va_m2))
    print(f"Fold {i}: M1 train={len(tr_m1)} val={len(va_m1)} | "
          f"M2 train={len(tr_m2)} val={len(va_m2)}")

print("\nSeluruh pemeriksaan kebocoran fold lulus.")

In [ ]:
# =============================================================
# CELL 9 - BASELINE MEAN-REVERSION
# =============================================================

def compute_mean_reversion_coeffs(
    speed: np.ndarray,
    train_origins: np.ndarray,
    history: int = 15,
    horizons: tuple = (5, 10, 15),
    stride: int = 30,
) -> Tuple[np.ndarray, Dict[int, np.ndarray]]:
    """Menghitung koefisien mean-reversion per ruas per horizon dari data train.

    Koefisien c dipilih agar
        prediksi = nilai_terakhir + c * (mean_long - nilai_terakhir)
    meminimalkan MSE pada sub-sampel train.
    """
    train_speed = speed[: train_origins[-1] + 1]
    valid       = np.where(train_speed > 0, train_speed, np.nan)
    long_mean   = np.nan_to_num(np.nanmean(valid, axis=0), nan=0.0).astype(np.float32)

    coeffs: Dict[int, np.ndarray] = {}
    for h in horizons:
        origins_sub = train_origins[::stride]
        last_vals   = speed[origins_sub]         # (W, N)
        targets     = speed[origins_sub + h]     # (W, N)
        delta       = long_mean[None, :] - last_vals
        num = (delta * (targets - last_vals)).sum(axis=0)
        den = (delta ** 2).sum(axis=0)
        c   = np.where(den > 1e-8, num / den, 0.0)
        coeffs[h] = np.clip(c, 0.0, 1.0).astype(np.float32)

    return long_mean, coeffs


def predict_mean_reversion(
    histories: np.ndarray,
    long_mean: np.ndarray,
    coeffs: Dict[int, np.ndarray],
    horizons: tuple = (5, 10, 15),
) -> np.ndarray:
    """Menghasilkan prediksi mean-reversion shape (W, len(horizons), N)."""
    last_val = histories[:, -1, :]          # (W, N)
    preds    = []
    for h in horizons:
        c = coeffs[h][None, :]              # (1, N)
        preds.append(last_val + c * (long_mean[None, :] - last_val))
    return np.stack(preds, axis=1)          # (W, 3, N)


def compose_prediction(baseline: np.ndarray, residual: np.ndarray) -> np.ndarray:
    """Menggabungkan baseline dan residual model."""
    return baseline + residual


# --- Skor baseline per fold shadow ---
print("=" * 60)
print("BASELINE MEAN-REVERSION")
print("=" * 60)

baseline_scores = []
for fold_idx in range(3):
    tr_m1, va_m1 = FOLDS_M1[fold_idx]
    tr_m2, va_m2 = FOLDS_M2[fold_idx]

    mean_m1_f, c_m1_f = compute_mean_reversion_coeffs(speed_m1, tr_m1, stride=30)
    mean_m2_f, c_m2_f = compute_mean_reversion_coeffs(speed_m2, tr_m2, stride=30)

    mse_per_horizon = []
    for h_idx, h in enumerate(HORIZONS):
        hist_m1_va, tgt_m1_va = make_windows(speed_m1, va_m1)
        pred_m1 = predict_mean_reversion(hist_m1_va, mean_m1_f, c_m1_f)
        err_m1  = float(((pred_m1[:, h_idx, :] - tgt_m1_va[:, h_idx, :]) ** 2).mean())

        hist_m2_va, tgt_m2_va = make_windows(speed_m2, va_m2)
        pred_m2 = predict_mean_reversion(hist_m2_va, mean_m2_f, c_m2_f)
        err_m2  = float(((pred_m2[:, h_idx, :] - tgt_m2_va[:, h_idx, :]) ** 2).mean())

        mse_combined = REGIME_WEIGHTS["m1"] * err_m1 + REGIME_WEIGHTS["m2"] * err_m2
        mse_per_horizon.append(mse_combined)

    mean_mse = float(np.mean(mse_per_horizon))
    baseline_scores.append(mean_mse)
    print(f"Fold {fold_idx}: h-MSE {[f'{v:.3f}' for v in mse_per_horizon]} | rata-rata={mean_mse:.4f}")

global_baseline_mse = float(np.mean(baseline_scores))
print(f"\nMSE mean-reversion global: {global_baseline_mse:.4f}")
if 35 <= global_baseline_mse <= 55:
    print("Baseline parity check: LULUS")
else:
    print(f"PERINGATAN: MSE {global_baseline_mse:.4f} di luar rentang wajar (35-55). Periksa target alignment.")

In [ ]:
# =============================================================
# CELL 10 - NORMALISASI DAN PRAKOMPUTASI FITUR GRAF
# =============================================================

def row_normalize(adj: np.ndarray) -> np.ndarray:
    """Normalisasi baris matriks adjacency (baris semua-nol tetap nol)."""
    row_sums = adj.sum(axis=1)
    row_sums = np.where(row_sums == 0, 1.0, row_sums)
    return (adj / row_sums[:, None]).astype(np.float32)


class SpeedNormalizer:
    """Normalisasi z-score per ruas, hanya dari data train."""

    def __init__(self, eps: float = 1.0):
        self.eps  = eps
        self.mean_: Optional[np.ndarray] = None
        self.std_:  Optional[np.ndarray] = None

    def fit(self, speed: np.ndarray) -> "SpeedNormalizer":
        valid = np.where(speed > 0, speed, np.nan)
        self.mean_ = np.nan_to_num(np.nanmean(valid, axis=0), nan=0.0).astype(np.float32)
        self.std_  = np.nan_to_num(np.nanstd(valid,  axis=0), nan=1.0).astype(np.float32)
        self.std_  = np.maximum(self.std_, self.eps)
        return self

    def transform(self, speed: np.ndarray) -> np.ndarray:
        result = (speed - self.mean_) / self.std_
        return np.where(speed == 0, 0.0, result).astype(np.float32)

    def inverse_transform(self, norm: np.ndarray) -> np.ndarray:
        return (norm * self.std_ + self.mean_).astype(np.float32)


# Matriks adjacency ternormalisasi
A_fwd_1 = row_normalize(adj_raw)
A_bwd_1 = row_normalize(adj_raw.T)
A_fwd_2 = row_normalize(A_fwd_1 @ A_fwd_1)
A_bwd_2 = row_normalize(A_bwd_1 @ A_bwd_1)

if CFG["USE_GRAPH"] and CFG["GRAPH_HOPS"] >= 2:
    A_FWD_HOPS = [A_fwd_1, A_fwd_2]
    A_BWD_HOPS = [A_bwd_1, A_bwd_2]
else:
    A_FWD_HOPS = [A_fwd_1]
    A_BWD_HOPS = [A_bwd_1]
N_HOPS = len(A_FWD_HOPS)

# Tensor versi untuk GPU
if TORCH_AVAILABLE:
    A_FWD_T = [torch.from_numpy(A).to(DEVICE) for A in A_FWD_HOPS]
    A_BWD_T = [torch.from_numpy(A).to(DEVICE) for A in A_BWD_HOPS]

print(f"Graf: {N_HOPS} hop maju dan mundur")
print(f"Normalisasi A_fwd_1 row-sum rata-rata: {A_fwd_1.sum(1).mean():.3f}")

# Verifikasi row_normalize
_adj_test = np.array([[1, 1, 0], [0, 1, 0], [0, 0, 0]], dtype=np.float32)
_norm_test = row_normalize(_adj_test)
np.testing.assert_allclose(_norm_test[:2].sum(axis=1), 1.0, rtol=1e-5)
np.testing.assert_array_equal(_norm_test[2], np.zeros(3, dtype=np.float32))
print("Verifikasi row_normalize: LULUS")

# Metadata statis ruas
if CFG["USE_ROAD_METADATA"]:
    road_len     = np.array([sum(s["length"] for s in r) for r in roads_meta], dtype=np.float32)
    road_nseg    = np.array([len(r)                       for r in roads_meta], dtype=np.float32)
    road_class   = np.array([r[0].get("roadclass",  0)   for r in roads_meta], dtype=np.float32)
    road_formway = np.array([r[0].get("formway",    0)   for r in roads_meta], dtype=np.float32)

    def _norm_stat(x: np.ndarray) -> np.ndarray:
        mu, sigma = x.mean(), x.std() + 1e-6
        return ((x - mu) / sigma).astype(np.float32)

    road_static = np.stack([
        _norm_stat(road_len), _norm_stat(road_nseg),
        _norm_stat(road_class), _norm_stat(road_formway),
    ], axis=1)  # (1260, 4)
else:
    road_static = np.zeros((N_ROADS, 0), dtype=np.float32)

N_STATIC = road_static.shape[1]
print(f"Metadata statis: {road_static.shape}")

In [ ]:
# =============================================================
# CELL 11 - DATASET LAZY DAN DATALOADER
# =============================================================

class LazyTrafficDataset(Dataset):
    """Dataset berbasis jendela untuk peramalan kecepatan lalu lintas.

    Setiap item berisi:
    - dynamic     : (N, D_dyn)  fitur statistik + hop agregasi (untuk Graph-STID)
    - history_norm: (N, T)      riwayat kecepatan ternormalisasi (untuk WaveNet)
    - baseline    : (N, H)      prediksi mean-reversion
    - target      : (N, H)      target skala asli (km/jam)
    - active_mask : (N,) bool   ruas aktif pada jendela ini
    """

    def __init__(
        self,
        speed_raw: np.ndarray,
        origins: np.ndarray,
        normalizer: SpeedNormalizer,
        long_mean: np.ndarray,
        mr_coeffs: Dict[int, np.ndarray],
        A_fwd_list: List[np.ndarray],
        A_bwd_list: List[np.ndarray],
        road_static_feats: np.ndarray,
        horizons: tuple = (5, 10, 15),
        history: int = 15,
        max_samples: Optional[int] = None,
    ):
        self.speed_raw  = speed_raw
        self.normalizer = normalizer
        self.long_mean  = long_mean
        self.mr_coeffs  = mr_coeffs
        self.A_fwd      = A_fwd_list
        self.A_bwd      = A_bwd_list
        self.road_static = road_static_feats
        self.horizons   = horizons
        self.history    = history

        self.origins = origins
        if max_samples is not None and len(origins) > max_samples:
            idx = np.linspace(0, len(origins) - 1, max_samples, dtype=int)
            self.origins = origins[idx]

    def __len__(self) -> int:
        return len(self.origins)

    def __getitem__(self, idx: int) -> Dict:
        origin = self.origins[idx]
        h_len  = self.history
        N      = self.speed_raw.shape[1]

        hist_idx   = np.arange(origin - h_len + 1, origin + 1)
        window_raw = self.speed_raw[hist_idx]          # (H, N)
        window_norm = self.normalizer.transform(window_raw)  # (H, N)

        # ---- Fitur statistik temporal (9 fitur per ruas) ----
        last_val = window_norm[-1]                           # (N,)
        mean3    = window_norm[-3:].mean(0)                  # (N,)
        mean5    = window_norm[-5:].mean(0)
        mean15   = window_norm.mean(0)
        diff1    = window_norm[-1] - window_norm[-2]
        diff5    = window_norm[-1] - window_norm[-5]
        std5     = window_norm[-5:].std(0)
        std15    = window_norm.std(0)
        global_m = np.full(N, mean15.mean(), dtype=np.float32)

        scalar_feats = np.stack(
            [last_val, mean3, mean5, mean15, diff1, diff5, std5, std15, global_m],
            axis=0,
        )  # (9, N)

        # ---- Agregasi tetangga (per hop, nilai timestep terakhir) ----
        hop_last = []
        for A_f, A_b in zip(self.A_fwd, self.A_bwd):
            hop_last.append(last_val @ A_f.T)  # (N,)
            hop_last.append(last_val @ A_b.T)  # (N,)
        hop_arr = np.stack(hop_last, axis=0)   # (n_hops*2, N)

        # ---- Dynamic features: (N, D_dyn) ----
        dynamic_feats = np.concatenate([scalar_feats, hop_arr], axis=0).T  # (N, 9+n_hops*2)

        # ---- Riwayat ternormalisasi untuk WaveNet: (N, T) ----
        history_norm = window_norm.T.astype(np.float32)  # (N, T)

        # ---- Baseline mean-reversion: (N, H) ----
        last_raw = window_raw[-1].astype(np.float32)     # (N,)
        bl_list  = []
        for h in self.horizons:
            c = self.mr_coeffs[h]
            bl_list.append(last_raw + c * (self.long_mean - last_raw))
        baseline = np.stack(bl_list, axis=1)             # (N, H)

        # ---- Target skala asli: (N, H) ----
        targets = np.stack(
            [self.speed_raw[origin + h] for h in self.horizons], axis=1
        ).astype(np.float32)  # (N, H)

        # ---- Mask ruas aktif ----
        active_mask = (window_raw > 0).any(axis=0)       # (N,) bool

        return {
            "dynamic"     : torch.from_numpy(dynamic_feats.astype(np.float32)),  # (N, D)
            "history_norm": torch.from_numpy(history_norm),                      # (N, T)
            "baseline"    : torch.from_numpy(baseline.astype(np.float32)),       # (N, H)
            "target"      : torch.from_numpy(targets),                           # (N, H)
            "active_mask" : torch.from_numpy(active_mask),                       # (N,) bool
        }


DYNAMIC_DIM = 9 + N_HOPS * 2
print(f"DYNAMIC_DIM={DYNAMIC_DIM} | STATIC_DIM={N_STATIC} | HIST_LEN={HIST_LEN}")

In [ ]:
# =============================================================
# CELL 12 - RESIDUAL GRAPH-STID
# =============================================================

class ResidualMLP(nn.Module):
    """Blok MLP residual dengan LayerNorm, GELU, dan dropout."""

    def __init__(self, dim: int, dropout: float = 0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fc1  = nn.Linear(dim, dim * 2)
        self.act  = nn.GELU()
        self.drop = nn.Dropout(dropout)
        self.fc2  = nn.Linear(dim * 2, dim)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x: "torch.Tensor") -> "torch.Tensor":
        h = self.norm(x)
        h = self.fc2(self.drop(self.act(self.fc1(h))))
        return x + self.drop2(h)


class ResidualGraphSTID(nn.Module):
    """Residual Graph-STID untuk peramalan kecepatan lalu lintas.

    Prediksi akhir = baseline_mean_reversion + output_head(features).
    Head output diinisialisasi nol sehingga pada awal pelatihan
    keluaran model identik dengan baseline.

    Args:
        roads        : jumlah ruas (1260)
        dynamic_dim  : dimensi fitur dinamis per ruas
        static_dim   : dimensi fitur statis per ruas
        hidden_dim   : dimensi tersembunyi
        road_embed_dim: dimensi embedding ruas
        horizons     : jumlah horizon prediksi
        dropout      : laju dropout
    """

    def __init__(
        self,
        roads: int,
        dynamic_dim: int,
        static_dim: int,
        hidden_dim: int,
        road_embed_dim: int,
        horizons: int = 3,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.roads   = roads
        self.horizons = horizons

        self.road_embed = nn.Embedding(roads, road_embed_dim)

        input_dim = dynamic_dim + road_embed_dim + max(static_dim, 1)
        self.input_proj = nn.Linear(input_dim, hidden_dim)

        self.mlp_blocks = nn.ModuleList([
            ResidualMLP(hidden_dim, dropout) for _ in range(3)
        ])

        # Head diinisialisasi nol -> prediksi awal == baseline
        self.output_head = nn.Linear(hidden_dim, horizons)
        nn.init.zeros_(self.output_head.weight)
        nn.init.zeros_(self.output_head.bias)

        self.register_buffer("road_ids", torch.arange(roads))

    def forward(
        self,
        dynamic : "torch.Tensor",
        static  : "torch.Tensor",
        baseline: "torch.Tensor",
    ) -> "torch.Tensor":
        """
        Args:
            dynamic : (B, N, D_dyn)
            static  : (N, S) - tidak ada dimensi batch
            baseline: (B, N, H)
        Returns:
            prediction: (B, N, H)
        """
        B, N, _ = dynamic.shape

        road_emb = self.road_embed(self.road_ids).unsqueeze(0).expand(B, -1, -1)  # (B, N, E)

        if static.shape[-1] > 0:
            static_exp = static.unsqueeze(0).expand(B, -1, -1)  # (B, N, S)
            x = torch.cat([dynamic, road_emb, static_exp], dim=-1)
        else:
            dummy = torch.zeros(B, N, 1, device=dynamic.device)
            x = torch.cat([dynamic, road_emb, dummy], dim=-1)

        x = self.input_proj(x)
        for block in self.mlp_blocks:
            x = block(x)

        residual = self.output_head(x)  # (B, N, H)
        return baseline + residual


# --- Verifikasi shape dan inisialisasi nol ---
if TORCH_AVAILABLE:
    _m = ResidualGraphSTID(
        roads=8, dynamic_dim=DYNAMIC_DIM, static_dim=N_STATIC,
        hidden_dim=32, road_embed_dim=8, horizons=3, dropout=0.0,
    )
    _d = torch.randn(2, 8, DYNAMIC_DIM)
    _s = torch.zeros(8, max(N_STATIC, 1))
    _b = torch.randn(2, 8, 3)
    _out = _m(_d, _s, _b)
    assert _out.shape == (2, 8, 3)
    torch.testing.assert_close(_out, _b, msg="Inisialisasi nol gagal")
    print("ResidualGraphSTID shape test       : LULUS")
    print("ResidualGraphSTID zero-init test   : LULUS")
    del _m, _d, _s, _b, _out

In [ ]:
# =============================================================
# CELL 13 - COMPACT GRAPH WAVENET
# =============================================================

class CompactGraphWaveNet(nn.Module):
    """Compact Graph WaveNet untuk peramalan kecepatan lalu lintas.

    Menerima riwayat kecepatan ternormalisasi (B, N, T) dan baseline
    mean-reversion (B, N, H). Menghasilkan prediksi (B, N, H).

    Arsitektur:
    - Proyeksi T -> C per ruas (Linear)
    - Beberapa lapisan gated MLP dengan agregasi graf opsional
    - Head output diinisialisasi nol (prediksi awal == baseline)

    Args:
        roads            : jumlah ruas
        history          : panjang riwayat T
        horizons         : jumlah horizon
        dilations        : jumlah lapisan (panjang tuple menentukan kedalaman)
        residual_channels: kanal residual C
        skip_channels    : kanal skip S
    """

    def __init__(
        self,
        roads: int,
        history: int = 15,
        horizons: int = 3,
        dilations: Tuple[int, ...] = (1, 2, 4, 8),
        residual_channels: int = 32,
        skip_channels: int = 64,
    ):
        super().__init__()
        self.roads    = roads
        self.horizons = horizons
        n_layers      = len(dilations)

        # Receptive field mencakup seluruh history (linear projection)
        self._receptive_field = history

        # Proyeksi history -> representasi C
        self.input_proj = nn.Linear(history, residual_channels)

        # Lapisan gated
        self.gate_proj  = nn.ModuleList([nn.Linear(residual_channels, residual_channels * 2) for _ in range(n_layers)])
        self.skip_proj  = nn.ModuleList([nn.Linear(residual_channels, skip_channels)          for _ in range(n_layers)])
        self.res_proj   = nn.ModuleList([nn.Linear(residual_channels, residual_channels)       for _ in range(n_layers)])

        # Head output diinisialisasi nol
        self.output_head = nn.Sequential(
            nn.ReLU(),
            nn.Linear(skip_channels, skip_channels),
            nn.ReLU(),
            nn.Linear(skip_channels, horizons),
        )
        for m in self.output_head.modules():
            if isinstance(m, nn.Linear):
                nn.init.zeros_(m.weight)
                nn.init.zeros_(m.bias)

    @property
    def receptive_field(self) -> int:
        return self._receptive_field

    def forward(
        self,
        x: "torch.Tensor",
        baseline: "torch.Tensor",
        A_fwd_list: Optional[List["torch.Tensor"]] = None,
        A_bwd_list: Optional[List["torch.Tensor"]] = None,
    ) -> "torch.Tensor":
        """
        Args:
            x        : (B, N, T) riwayat kecepatan ternormalisasi
            baseline : (B, N, H) prediksi baseline
            A_fwd_list: list tensor adjacency maju (N, N) opsional
            A_bwd_list: list tensor adjacency mundur (N, N) opsional
        Returns:
            prediction: (B, N, H)
        """
        # h: (B, N, C)
        h = self.input_proj(x)

        skip_sum = None
        for gate_l, skip_l, res_l in zip(self.gate_proj, self.skip_proj, self.res_proj):
            # Gated activation
            g   = gate_l(h)          # (B, N, 2C)
            C2  = g.shape[-1] // 2
            h_g = torch.sigmoid(g[..., :C2]) * torch.tanh(g[..., C2:])  # (B, N, C)

            # Agregasi graf opsional (hop 1 forward)
            if A_fwd_list is not None:
                A = A_fwd_list[0]    # (N, N)
                # h_g: (B, N, C) -> untuk setiap b: A @ h_g[b] = (N, N)@(N,C)=(N,C)
                h_graph = torch.einsum("nk,bkc->bnc", A, h_g)  # (B, N, C)
                h_g = h_g + h_graph

            skip = skip_l(h_g)       # (B, N, S)
            skip_sum = skip if skip_sum is None else skip_sum + skip

            h = h + res_l(h_g)       # (B, N, C) residual

        residual = self.output_head(skip_sum)  # (B, N, H)
        return baseline + residual


# --- Verifikasi shape dan receptive field ---
if TORCH_AVAILABLE:
    _wn = CompactGraphWaveNet(
        roads=8, history=HIST_LEN, horizons=3,
        dilations=(1, 2, 4, 8), residual_channels=8, skip_channels=16,
    )
    assert _wn.receptive_field >= HIST_LEN, (
        f"Receptive field ({_wn.receptive_field}) < HIST_LEN ({HIST_LEN})"
    )
    _x  = torch.randn(2, 8, HIST_LEN)
    _bl = torch.randn(2, 8, 3)
    _out = _wn(_x, _bl)
    assert _out.shape == (2, 8, 3)
    print(f"CompactGraphWaveNet receptive_field={_wn.receptive_field} (>={HIST_LEN}): LULUS")
    print(f"CompactGraphWaveNet shape test: LULUS")
    del _wn, _x, _bl, _out

In [ ]:
# =============================================================
# CELL 14 - MESIN PELATIHAN
# =============================================================

@dataclass
class TrainingResult:
    history: List[Dict] = field(default_factory=list)
    best_val_mse: float = float("inf")
    best_epoch: int = 0
    checkpoint_path: str = ""


def _forward_model(
    model: nn.Module,
    batch: Dict,
    static_t: "torch.Tensor",
    a_fwd: Optional[List["torch.Tensor"]] = None,
    a_bwd: Optional[List["torch.Tensor"]] = None,
) -> "torch.Tensor":
    """Dispatcher forward yang menangani kedua jenis model.

    ResidualGraphSTID : menerima (dynamic, static, baseline)
    CompactGraphWaveNet: menerima (history_norm, baseline, a_fwd, a_bwd)
    """
    if isinstance(model, ResidualGraphSTID):
        return model(
            batch["dynamic"],
            static_t,
            batch["baseline"],
        )
    elif isinstance(model, CompactGraphWaveNet):
        return model(
            batch["history_norm"],
            batch["baseline"],
            a_fwd,
            a_bwd,
        )
    else:
        raise TypeError(f"Tipe model tidak dikenal: {type(model)}")


def _make_scaler():
    """Membuat GradScaler dengan compat PyTorch 1.x dan 2.x."""
    try:
        return torch.amp.GradScaler("cuda")   # PyTorch >= 2.0
    except TypeError:
        return torch.cuda.amp.GradScaler()     # PyTorch < 2.0


def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    static_features: "torch.Tensor",
    epochs: int,
    patience: int,
    amp: bool,
    learning_rate: float,
    weight_decay: float,
    grad_clip: float,
    checkpoint_path,
    resume: bool = False,
    clip_range: Tuple[float, float] = (0.0, 160.0),
    a_fwd: Optional[List["torch.Tensor"]] = None,
    a_bwd: Optional[List["torch.Tensor"]] = None,
) -> TrainingResult:
    """Melatih model dengan AdamW, AMP, cosine LR decay, dan early stopping."""
    checkpoint_path  = Path(checkpoint_path)
    model            = model.to(DEVICE)
    static_features  = static_features.to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=learning_rate, weight_decay=weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(epochs, 1), eta_min=learning_rate * 0.01,
    )
    use_amp = amp and DEVICE is not None and DEVICE.type == "cuda"
    scaler  = _make_scaler() if use_amp else None

    result    = TrainingResult(checkpoint_path=str(checkpoint_path))
    start_ep  = 0
    no_improve = 0

    if resume and checkpoint_path.exists():
        ckpt = torch.load(checkpoint_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        scheduler.load_state_dict(ckpt["scheduler_state"])
        if scaler and "scaler_state" in ckpt:
            scaler.load_state_dict(ckpt["scaler_state"])
        start_ep          = ckpt["epoch"] + 1
        result.best_val_mse = ckpt["best_val_mse"]
        result.best_epoch   = ckpt["best_epoch"]
        result.history      = ckpt.get("history", [])
        no_improve          = ckpt.get("no_improve", 0)
        print(f"Melanjutkan dari epoch {start_ep}, best_mse={result.best_val_mse:.4f}")

    for epoch in range(start_ep, epochs):
        # ---------- Train ----------
        model.train()
        total_loss, n_batches = 0.0, 0
        t_ep = time.time()

        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()

            if use_amp:
                with torch.amp.autocast(device_type="cuda"):
                    pred = _forward_model(model, batch, static_features, a_fwd, a_bwd)
                    mask = batch["active_mask"].unsqueeze(-1).float()
                    loss = ((pred - batch["target"]) ** 2 * mask).sum() / (
                        mask.sum() * pred.shape[-1] + 1e-8
                    )
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                pred = _forward_model(model, batch, static_features, a_fwd, a_bwd)
                mask = batch["active_mask"].unsqueeze(-1).float()
                loss = ((pred - batch["target"]) ** 2 * mask).sum() / (
                    mask.sum() * pred.shape[-1] + 1e-8
                )
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()

            total_loss += loss.item()
            n_batches  += 1

        scheduler.step()
        train_loss = total_loss / max(n_batches, 1)

        # ---------- Validasi ----------
        model.eval()
        val_preds, val_tgts = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                pred  = _forward_model(model, batch, static_features, a_fwd, a_bwd)
                val_preds.append(pred.cpu().numpy())
                val_tgts.append(batch["target"].cpu().numpy())

        val_pred_np = np.clip(
            np.concatenate(val_preds, axis=0),
            clip_range[0], clip_range[1],
        )
        val_tgt_np  = np.concatenate(val_tgts, axis=0)
        val_mse     = float(np.mean((val_pred_np - val_tgt_np) ** 2))
        ep_time     = time.time() - t_ep

        result.history.append({
            "epoch": epoch, "train_loss": train_loss,
            "val_mse": val_mse, "time_s": ep_time,
        })

        improved = val_mse < result.best_val_mse
        print(f"Epoch {epoch:3d} | loss={train_loss:.4f} | val_mse={val_mse:.4f}"
              f"{' *' if improved else '  '} | {ep_time:.1f}s")

        if improved:
            result.best_val_mse = val_mse
            result.best_epoch   = epoch
            no_improve = 0
            torch.save({
                "epoch": epoch, "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scheduler_state": scheduler.state_dict(),
                "scaler_state"   : scaler.state_dict() if scaler else {},
                "best_val_mse"   : result.best_val_mse,
                "best_epoch"     : result.best_epoch,
                "history"        : result.history,
                "no_improve"     : no_improve,
                "cfg"            : CFG,
            }, checkpoint_path)
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping epoch {epoch} (no_improve={no_improve})")
                break

    if checkpoint_path.exists():
        ckpt = torch.load(checkpoint_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"])
        print(f"Model terbaik dimuat dari epoch {result.best_epoch}, mse={result.best_val_mse:.4f}")

    return result


print("Mesin pelatihan didefinisikan.")

In [ ]:
# =============================================================
# CELL 15 - EVALUASI MSE SKALA ASLI
# =============================================================

def evaluate_mse(
    model: nn.Module,
    speed_raw: np.ndarray,
    val_origins: np.ndarray,
    normalizer: SpeedNormalizer,
    long_mean: np.ndarray,
    mr_coeffs: dict,
    always_zero_mask: np.ndarray,
    regime_label: str,
    a_fwd: Optional[List] = None,
    a_bwd: Optional[List] = None,
) -> Dict:
    """Mengevaluasi MSE pada skala kecepatan asli (km/jam).

    Returns:
        dict: overall, h5, h10, h15, baseline_overall, h{n}_baseline
    """
    model.eval()
    static_t = torch.from_numpy(road_static).to(DEVICE)

    ds = LazyTrafficDataset(
        speed_raw=speed_raw, origins=val_origins,
        normalizer=normalizer, long_mean=long_mean, mr_coeffs=mr_coeffs,
        A_fwd_list=A_FWD_HOPS, A_bwd_list=A_BWD_HOPS,
        road_static_feats=road_static, horizons=HORIZONS, history=HIST_LEN,
    )
    loader = DataLoader(ds, batch_size=CFG["BATCH_SIZE"] * 2, shuffle=False, num_workers=0)

    model_preds, mr_preds, all_targets = [], [], []

    with torch.no_grad():
        for batch in loader:
            batch_dev = {k: v.to(DEVICE) for k, v in batch.items()}
            pred      = _forward_model(model, batch_dev, static_t, a_fwd, a_bwd)
            pred_np   = np.clip(pred.cpu().numpy(),
                                CFG["PREDICTION_CLIP"][0], CFG["PREDICTION_CLIP"][1])
            bl_np     = batch["baseline"].numpy()

            # Terapkan mask ruas yang selalu nol
            pred_np[:, always_zero_mask, :] = 0.0
            bl_np[:, always_zero_mask, :]   = 0.0

            model_preds.append(pred_np)
            mr_preds.append(bl_np)
            all_targets.append(batch["target"].numpy())

    mp  = np.concatenate(model_preds, axis=0)   # (W, N, H)
    bl  = np.concatenate(mr_preds,    axis=0)
    tgt = np.concatenate(all_targets, axis=0)

    results = {
        "overall"          : float(np.mean((mp  - tgt) ** 2)),
        "baseline_overall" : float(np.mean((bl  - tgt) ** 2)),
    }
    for h_idx, h in enumerate(HORIZONS):
        results[f"h{h}"]          = float(np.mean((mp[:, :, h_idx]  - tgt[:, :, h_idx]) ** 2))
        results[f"h{h}_baseline"] = float(np.mean((bl[:, :, h_idx]  - tgt[:, :, h_idx]) ** 2))

    print(f"  [{regime_label}] Model={results['overall']:.4f} | "
          f"Baseline={results['baseline_overall']:.4f} | "
          f"Delta={results['overall'] - results['baseline_overall']:+.4f}")
    for h in HORIZONS:
        print(f"    h{h}: {results[f'h{h}']:.4f} (bl: {results[f'h{h}_baseline']:.4f})")

    return results


print("Fungsi evaluate_mse didefinisikan.")

In [ ]:
# =============================================================
# CELL 16 - EKSEKUSI SMOKE / CV
# =============================================================

RUN_MODE    = CFG["RUN_MODE"]
MODEL_NAME  = CFG["MODEL"]

SMOKE_MAX_SAMPLES = 512
SMOKE_EPOCHS      = 2

FOLDS_TO_RUN = CFG["FOLDS"] if RUN_MODE != "smoke" else [max(CFG["FOLDS"])]
print(f"Mode: {RUN_MODE} | Model: {MODEL_NAME} | Folds: {FOLDS_TO_RUN}")
print("=" * 60)

if not TORCH_AVAILABLE:
    print("PyTorch tidak tersedia. Cell ini dilewati.")
    cv_results = []
    LAST_INFER_MODEL = None
    LAST_NORM_M1 = LAST_NORM_M2 = None
    LAST_MEAN_M1 = LAST_MEAN_M2 = None
    LAST_COEFFS_M1 = LAST_COEFFS_M2 = None
else:
    # CELL 16 - EKSEKUSI SMOKE / CV
    # =============================================================
    
    RUN_MODE    = CFG["RUN_MODE"]
    MODEL_NAME  = CFG["MODEL"]
    
    SMOKE_MAX_SAMPLES = 512
    SMOKE_EPOCHS      = 2
    
    FOLDS_TO_RUN = CFG["FOLDS"] if RUN_MODE != "smoke" else [max(CFG["FOLDS"])]
    print(f"Mode: {RUN_MODE} | Model: {MODEL_NAME} | Folds: {FOLDS_TO_RUN}")
    print("=" * 60)
    
    cv_results           = []
    best_epochs_per_fold = []
    LAST_INFER_MODEL     = None
    LAST_NORM_M1         = None
    LAST_NORM_M2         = None
    LAST_MEAN_M1         = None
    LAST_MEAN_M2         = None
    LAST_COEFFS_M1       = None
    LAST_COEFFS_M2       = None
    
    for fold_idx in FOLDS_TO_RUN:
        print(f"\n{'=' * 60}")
        print(f"FOLD {fold_idx}")
        print(f"{'=' * 60}")
    
        tr_m1, va_m1 = FOLDS_M1[fold_idx]
        tr_m2, va_m2 = FOLDS_M2[fold_idx]
    
        max_train = SMOKE_MAX_SAMPLES if RUN_MODE == "smoke" else None
        epochs    = SMOKE_EPOCHS      if RUN_MODE == "smoke" else CFG["EPOCHS"]
    
        # Fit normalizer dan koefisien HANYA dari train
        norm_m1 = SpeedNormalizer().fit(speed_m1[: tr_m1[-1] + 1])
        norm_m2 = SpeedNormalizer().fit(speed_m2[: tr_m2[-1] + 1])
        mean_m1, c_m1 = compute_mean_reversion_coeffs(speed_m1, tr_m1, stride=30)
        mean_m2, c_m2 = compute_mean_reversion_coeffs(speed_m2, tr_m2, stride=30)
    
        # Dataset gabungan
        per_regime = (max_train // 2) if max_train else None
        ds_train = ConcatDataset([
            LazyTrafficDataset(speed_m1, tr_m1, norm_m1, mean_m1, c_m1,
                               A_FWD_HOPS, A_BWD_HOPS, road_static, HORIZONS, HIST_LEN,
                               max_samples=per_regime),
            LazyTrafficDataset(speed_m2, tr_m2, norm_m2, mean_m2, c_m2,
                               A_FWD_HOPS, A_BWD_HOPS, road_static, HORIZONS, HIST_LEN,
                               max_samples=per_regime),
        ])
        ds_val = ConcatDataset([
            LazyTrafficDataset(speed_m1, va_m1, norm_m1, mean_m1, c_m1,
                               A_FWD_HOPS, A_BWD_HOPS, road_static, HORIZONS, HIST_LEN),
            LazyTrafficDataset(speed_m2, va_m2, norm_m2, mean_m2, c_m2,
                               A_FWD_HOPS, A_BWD_HOPS, road_static, HORIZONS, HIST_LEN),
        ])
    
        loader_train = DataLoader(
            ds_train, batch_size=CFG["BATCH_SIZE"], shuffle=True,
            num_workers=CFG["NUM_WORKERS"],
            pin_memory=(DEVICE is not None and DEVICE.type == "cuda"), drop_last=True,
        )
        loader_val = DataLoader(
            ds_val, batch_size=CFG["BATCH_SIZE"] * 2, shuffle=False,
            num_workers=CFG["NUM_WORKERS"],
        )
    
        print(f"Train windows: {len(ds_train)} | Val windows: {len(ds_val)}")
    
        # Bangun model
        static_t = torch.from_numpy(road_static).to(DEVICE)
        if N_STATIC == 0:
            static_t = torch.zeros(N_ROADS, 1, device=DEVICE)
    
        if MODEL_NAME == "graph_stid":
            model = ResidualGraphSTID(
                roads=N_ROADS,
                dynamic_dim=DYNAMIC_DIM,
                static_dim=max(N_STATIC, 1),
                hidden_dim=CFG["HIDDEN_DIM"],
                road_embed_dim=CFG["ROAD_EMBED_DIM"],
                horizons=len(HORIZONS),
                dropout=CFG["DROPOUT"],
            )
            a_fwd_arg, a_bwd_arg = None, None
        elif MODEL_NAME == "graph_wavenet":
            model = CompactGraphWaveNet(
                roads=N_ROADS, history=HIST_LEN, horizons=len(HORIZONS),
                dilations=(1, 2, 4, 8), residual_channels=32, skip_channels=64,
            )
            a_fwd_arg = A_FWD_T
            a_bwd_arg = A_BWD_T
        else:
            raise ValueError(f"Model tidak dikenal: {MODEL_NAME}")
    
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Model: {MODEL_NAME} | Parameter: {n_params:,}")
    
        ckpt_path = WORK_DIR / "outputs" / f"best_fold{fold_idx}_{MODEL_NAME}.pt"
    
        result = train_model(
            model=model, train_loader=loader_train, val_loader=loader_val,
            static_features=static_t, epochs=epochs, patience=CFG["PATIENCE"],
            amp=CFG["AMP"], learning_rate=CFG["LR"], weight_decay=CFG["WEIGHT_DECAY"],
            grad_clip=CFG["GRAD_CLIP"], checkpoint_path=ckpt_path,
            clip_range=tuple(CFG["PREDICTION_CLIP"]),
            a_fwd=a_fwd_arg, a_bwd=a_bwd_arg,
        )
        best_epochs_per_fold.append(result.best_epoch)
    
        # Evaluasi per regime
        print(f"\nEvaluasi fold {fold_idx}:")
        res_m1 = evaluate_mse(model, speed_m1, va_m1, norm_m1, mean_m1, c_m1,
                              always_zero_m1, "M1", a_fwd_arg, a_bwd_arg)
        res_m2 = evaluate_mse(model, speed_m2, va_m2, norm_m2, mean_m2, c_m2,
                              always_zero_m2, "M2", a_fwd_arg, a_bwd_arg)
    
        combined_mse      = REGIME_WEIGHTS["m1"] * res_m1["overall"] + REGIME_WEIGHTS["m2"] * res_m2["overall"]
        combined_baseline = REGIME_WEIGHTS["m1"] * res_m1["baseline_overall"] + REGIME_WEIGHTS["m2"] * res_m2["baseline_overall"]
    
        print(f"\n  Fold {fold_idx} GABUNGAN: MSE={combined_mse:.4f} | Baseline={combined_baseline:.4f} | "
              f"Delta={combined_mse - combined_baseline:+.4f}")
    
        cv_results.append({
            "fold": fold_idx, "model": MODEL_NAME,
            "best_epoch": result.best_epoch, "best_val_mse": result.best_val_mse,
            "combined_mse": combined_mse, "combined_baseline": combined_baseline,
            "beats_baseline": combined_mse < combined_baseline,
            "m1": res_m1, "m2": res_m2, "checkpoint": str(ckpt_path),
        })
    
        # Simpan referensi model dan normalizer terakhir untuk inferensi
        LAST_INFER_MODEL = model
        LAST_NORM_M1, LAST_NORM_M2 = norm_m1, norm_m2
        LAST_MEAN_M1, LAST_MEAN_M2 = mean_m1, mean_m2
        LAST_COEFFS_M1, LAST_COEFFS_M2 = c_m1, c_m2
    
    # Simpan hasil CV
    cv_path = WORK_DIR / "outputs" / "notebook_cv_results.json"
    with open(cv_path, "w") as f:
        json.dump({"cv_results": cv_results, "cfg": CFG}, f, indent=2)
    print(f"\nHasil CV disimpan ke: {cv_path}")

In [ ]:
# =============================================================
# CELL 17 - RINGKASAN RANKING DAN ACCEPTANCE GATE
# =============================================================

print("=" * 60)
print("RINGKASAN CV DAN ACCEPTANCE GATE")
print("=" * 60)

if not cv_results:
    print("Tidak ada hasil CV.")
else:
    all_mse        = [r["combined_mse"]      for r in cv_results]
    all_baseline   = [r["combined_baseline"] for r in cv_results]
    n_beats        = sum(1 for r in cv_results if r["beats_baseline"])
    mean_mse       = float(np.mean(all_mse))
    std_mse        = float(np.std(all_mse))

    print(f"\nFold MSE:")
    for r in cv_results:
        print(f"  Fold {r['fold']}: {r['combined_mse']:.4f} "
              f"(baseline: {r['combined_baseline']:.4f}) "
              f"[{'LULUS' if r['beats_baseline'] else 'GAGAL'}]")
    print(f"\nRata-rata MSE: {mean_mse:.4f} +/- {std_mse:.4f}")
    print(f"Fold yang mengalahkan baseline: {n_beats}/{len(cv_results)}")

    # MSE per horizon
    print("\nMSE per horizon (rata-rata fold):")
    for h in HORIZONS:
        h_m1  = np.mean([r["m1"].get(f"h{h}", 0) for r in cv_results])
        h_m2  = np.mean([r["m2"].get(f"h{h}", 0) for r in cv_results])
        h_bl1 = np.mean([r["m1"].get(f"h{h}_baseline", 0) for r in cv_results])
        h_bl2 = np.mean([r["m2"].get(f"h{h}_baseline", 0) for r in cv_results])
        h_comb = REGIME_WEIGHTS["m1"] * h_m1  + REGIME_WEIGHTS["m2"] * h_m2
        h_bl   = REGIME_WEIGHTS["m1"] * h_bl1 + REGIME_WEIGHTS["m2"] * h_bl2
        print(f"  h{h}: {h_comb:.4f} (baseline: {h_bl:.4f}) delta: {h_comb-h_bl:+.4f}")

    # Acceptance Gate
    print("\n" + "-" * 40)
    print("ACCEPTANCE GATE")
    print("-" * 40)
    gate = {}

    gate["mse_below_39"] = mean_mse < 39.00
    print(f"[{'OK' if gate['mse_below_39'] else 'FAIL'}] 1. MSE rata-rata ({mean_mse:.4f}) < 39.00")

    gate["beats_2of3"] = n_beats >= 2
    print(f"[{'OK' if gate['beats_2of3'] else 'FAIL'}] 2. Mengalahkan baseline {n_beats}/{len(cv_results)} fold")

    h15_m = np.mean([REGIME_WEIGHTS["m1"]*r["m1"].get("h15",999)+REGIME_WEIGHTS["m2"]*r["m2"].get("h15",999) for r in cv_results])
    h15_b = np.mean([REGIME_WEIGHTS["m1"]*r["m1"].get("h15_baseline",0)+REGIME_WEIGHTS["m2"]*r["m2"].get("h15_baseline",0) for r in cv_results])
    gate["h15_ok"] = h15_m <= h15_b
    print(f"[{'OK' if gate['h15_ok'] else 'FAIL'}] 3. MSE h15 ({h15_m:.4f}) <= baseline ({h15_b:.4f})")

    regime_ok = True
    for rk in ["m1", "m2"]:
        rm = np.mean([r[rk]["overall"] for r in cv_results])
        rb = np.mean([r[rk]["baseline_overall"] for r in cv_results])
        if rm > rb * 1.01:
            regime_ok = False
            print(f"  PERINGATAN: {rk.upper()} memburuk > 1%")
    gate["regime_ok"] = regime_ok
    print(f"[{'OK' if gate['regime_ok'] else 'FAIL'}] 4. MSE per regime tidak memburuk > 1%")

    gate["fold_var"] = std_mse > 0
    print(f"[{'OK' if gate['fold_var'] else 'FAIL'}] 5. Variasi fold dilaporkan (std={std_mse:.4f})")

    SUBMIT_CANDIDATE = all(gate.values())
    print(f"\n{'=' * 40}")
    print(f"SUBMIT_CANDIDATE = {SUBMIT_CANDIDATE}")
    if not SUBMIT_CANDIDATE:
        print(f"Syarat belum terpenuhi: {[k for k,v in gate.items() if not v]}")
    print(f"{'=' * 40}")

In [ ]:
# =============================================================
# CELL 18 - PELATIHAN FINAL
# =============================================================
if not TORCH_AVAILABLE:
    print("PyTorch tidak tersedia. Pelatihan final dilewati.")
else:
    
    FINAL_MODEL    = None
    FINAL_NORM_M1  = None
    FINAL_NORM_M2  = None
    FINAL_MEAN_M1  = None
    FINAL_MEAN_M2  = None
    FINAL_COEFFS_M1 = None
    FINAL_COEFFS_M2 = None
    
    if RUN_MODE == "final":
        print("=" * 60)
        print("PELATIHAN FINAL PADA SELURUH DATA TRAIN")
        print("=" * 60)
    
        final_epochs = int(np.median(best_epochs_per_fold)) + 5 if best_epochs_per_fold else CFG["EPOCHS"]
        print(f"Epoch final: {final_epochs}")
    
        FINAL_NORM_M1 = SpeedNormalizer().fit(speed_m1)
        FINAL_NORM_M2 = SpeedNormalizer().fit(speed_m2)
        all_tr_m1 = np.arange(HIST_LEN - 1, len(speed_m1) - max(HORIZONS))
        all_tr_m2 = np.arange(HIST_LEN - 1, len(speed_m2) - max(HORIZONS))
        FINAL_MEAN_M1, FINAL_COEFFS_M1 = compute_mean_reversion_coeffs(speed_m1, all_tr_m1, stride=30)
        FINAL_MEAN_M2, FINAL_COEFFS_M2 = compute_mean_reversion_coeffs(speed_m2, all_tr_m2, stride=30)
    
        ds_final = ConcatDataset([
            LazyTrafficDataset(speed_m1, all_tr_m1, FINAL_NORM_M1, FINAL_MEAN_M1, FINAL_COEFFS_M1,
                               A_FWD_HOPS, A_BWD_HOPS, road_static, HORIZONS, HIST_LEN),
            LazyTrafficDataset(speed_m2, all_tr_m2, FINAL_NORM_M2, FINAL_MEAN_M2, FINAL_COEFFS_M2,
                               A_FWD_HOPS, A_BWD_HOPS, road_static, HORIZONS, HIST_LEN),
        ])
        loader_final = DataLoader(
            ds_final, batch_size=CFG["BATCH_SIZE"], shuffle=True,
            num_workers=CFG["NUM_WORKERS"], pin_memory=(DEVICE is not None and DEVICE.type=="cuda"), drop_last=True,
        )
    
        # Loader monitor kecil (tidak ada early stopping pada final)
        mon_orig = np.arange(len(speed_m2) - 200, len(speed_m2) - max(HORIZONS))
        ds_mon   = LazyTrafficDataset(speed_m2, mon_orig, FINAL_NORM_M2, FINAL_MEAN_M2, FINAL_COEFFS_M2,
                                      A_FWD_HOPS, A_BWD_HOPS, road_static, HORIZONS, HIST_LEN)
        loader_mon = DataLoader(ds_mon, batch_size=32, shuffle=False)
    
        static_t = torch.from_numpy(road_static).to(DEVICE)
        if N_STATIC == 0:
            static_t = torch.zeros(N_ROADS, 1, device=DEVICE)
    
        if MODEL_NAME == "graph_stid":
            FINAL_MODEL   = ResidualGraphSTID(
                roads=N_ROADS, dynamic_dim=DYNAMIC_DIM, static_dim=max(N_STATIC, 1),
                hidden_dim=CFG["HIDDEN_DIM"], road_embed_dim=CFG["ROAD_EMBED_DIM"],
                horizons=len(HORIZONS), dropout=CFG["DROPOUT"],
            )
            a_fwd_f, a_bwd_f = None, None
        else:
            FINAL_MODEL   = CompactGraphWaveNet(
                roads=N_ROADS, history=HIST_LEN, horizons=len(HORIZONS),
                dilations=(1, 2, 4, 8), residual_channels=32, skip_channels=64,
            )
            a_fwd_f, a_bwd_f = A_FWD_T, A_BWD_T
    
        final_ckpt = WORK_DIR / "outputs" / f"final_{MODEL_NAME}.pt"
        train_model(
            model=FINAL_MODEL, train_loader=loader_final, val_loader=loader_mon,
            static_features=static_t, epochs=final_epochs, patience=final_epochs,
            amp=CFG["AMP"], learning_rate=CFG["LR"], weight_decay=CFG["WEIGHT_DECAY"],
            grad_clip=CFG["GRAD_CLIP"], checkpoint_path=final_ckpt,
            clip_range=tuple(CFG["PREDICTION_CLIP"]),
            a_fwd=a_fwd_f, a_bwd=a_bwd_f,
        )
        print(f"Pelatihan final selesai. Model disimpan ke: {final_ckpt}")
    else:
        print(f"RUN_MODE='{RUN_MODE}'. Atur 'final' untuk pelatihan pada seluruh data.")

In [ ]:
# =============================================================
# CELL 19 - INFERENSI TEST DAN ROUTING REGIME
# =============================================================
# Ensure fallback variables exist (jika cell 16/18 dilewati)
try:
    _ = FINAL_MODEL
except NameError:
    FINAL_MODEL = None
    FINAL_NORM_M1 = FINAL_NORM_M2 = None
    FINAL_MEAN_M1 = FINAL_MEAN_M2 = None
    FINAL_COEFFS_M1 = FINAL_COEFFS_M2 = None
try:
    _ = LAST_INFER_MODEL
except NameError:
    LAST_INFER_MODEL = None
    LAST_NORM_M1 = LAST_NORM_M2 = None
    LAST_MEAN_M1 = LAST_MEAN_M2 = None
    LAST_COEFFS_M1 = LAST_COEFFS_M2 = None
try:
    _ = RUN_MODE
except NameError:
    RUN_MODE = CFG["RUN_MODE"]
    MODEL_NAME = CFG["MODEL"]


def build_sample_features(
    window_raw: np.ndarray,
    normalizer: SpeedNormalizer,
    long_mean: np.ndarray,
    mr_coeffs: Dict[int, np.ndarray],
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Membangun fitur untuk satu sampel test.

    Returns:
        dynamic     : (N, D_dyn)
        history_norm: (N, T)
        baseline    : (N, H)
        last_raw    : (N,)
    """
    N = window_raw.shape[1]
    window_norm = normalizer.transform(window_raw)  # (T, N)

    last_val = window_norm[-1]
    mean3    = window_norm[-3:].mean(0)
    mean5    = window_norm[-5:].mean(0)
    mean15   = window_norm.mean(0)
    diff1    = window_norm[-1] - window_norm[-2]
    diff5    = window_norm[-1] - window_norm[-5]
    std5     = window_norm[-5:].std(0)
    std15    = window_norm.std(0)
    global_m = np.full(N, mean15.mean(), dtype=np.float32)

    scalar_feats = np.stack(
        [last_val, mean3, mean5, mean15, diff1, diff5, std5, std15, global_m], axis=0,
    )  # (9, N)

    hop_last = []
    for A_f, A_b in zip(A_FWD_HOPS, A_BWD_HOPS):
        hop_last.append(last_val @ A_f.T)
        hop_last.append(last_val @ A_b.T)
    hop_arr = np.stack(hop_last, axis=0)  # (n_hops*2, N)

    dynamic      = np.concatenate([scalar_feats, hop_arr], axis=0).T.astype(np.float32)  # (N, D)
    history_norm = window_norm.T.astype(np.float32)                                       # (N, T)

    last_raw = window_raw[-1].astype(np.float32)
    bl_list  = []
    for h in HORIZONS:
        c = mr_coeffs[h]
        bl_list.append(last_raw + c * (long_mean - last_raw))
    baseline = np.stack(bl_list, axis=1).astype(np.float32)  # (N, H)

    return dynamic, history_norm, baseline, last_raw


def inference_test(
    model: nn.Module,
    test_hist_np: np.ndarray,
    test_regimes_np: np.ndarray,
    norm_m1: SpeedNormalizer,
    norm_m2: SpeedNormalizer,
    mean_m1: np.ndarray,
    mean_m2: np.ndarray,
    coeffs_m1: dict,
    coeffs_m2: dict,
    az_m1: np.ndarray,
    az_m2: np.ndarray,
    a_fwd: Optional[List] = None,
    a_bwd: Optional[List] = None,
    batch_size: int = 32,
) -> np.ndarray:
    """Inferensi test dengan routing per regime. Returns (540, 3, 1260)."""
    N_s  = len(test_hist_np)
    H    = len(HORIZONS)
    pred_cube = np.zeros((N_s, H, N_ROADS), dtype=np.float32)

    model.eval()
    static_t = torch.from_numpy(road_static).to(DEVICE)
    if N_STATIC == 0:
        static_t = torch.zeros(N_ROADS, 1, device=DEVICE)

    with torch.no_grad():
        for start in range(0, N_s, batch_size):
            end = min(start + batch_size, N_s)
            dyn_batch, hn_batch, bl_batch = [], [], []
            az_list = []

            for i in range(start, end):
                window  = test_hist_np[i]   # (T, N)
                regime  = test_regimes_np[i]
                norm    = norm_m1  if regime == "m1" else norm_m2
                lm      = mean_m1  if regime == "m1" else mean_m2
                coeffs  = coeffs_m1 if regime == "m1" else coeffs_m2
                az      = az_m1    if regime == "m1" else az_m2

                dyn, hn, bl, _ = build_sample_features(window, norm, lm, coeffs)
                dyn_batch.append(dyn)
                hn_batch.append(hn)
                bl_batch.append(bl)
                az_list.append(az)

            dyn_t = torch.from_numpy(np.stack(dyn_batch)).to(DEVICE)  # (B, N, D)
            hn_t  = torch.from_numpy(np.stack(hn_batch)).to(DEVICE)   # (B, N, T)
            bl_t  = torch.from_numpy(np.stack(bl_batch)).to(DEVICE)   # (B, N, H)

            batch = {
                "dynamic"     : dyn_t,
                "history_norm": hn_t,
                "baseline"    : bl_t,
                "active_mask" : torch.ones(len(dyn_batch), N_ROADS, dtype=torch.bool, device=DEVICE),
            }
            pred = _forward_model(model, batch, static_t, a_fwd, a_bwd)  # (B, N, H)
            pred_np = np.clip(pred.cpu().numpy(),
                              CFG["PREDICTION_CLIP"][0], CFG["PREDICTION_CLIP"][1])

            # Terapkan mask selalu nol per sampel
            for j, az in enumerate(az_list):
                pred_np[j, az, :] = 0.0

            pred_cube[start:end] = pred_np.transpose(0, 2, 1)  # (B, H, N)

    return pred_cube


# --- Pilih model untuk inferensi ---
PRED_CUBE = None

if not TORCH_AVAILABLE:
    infer_model = None
    print("PyTorch tidak tersedia. Menggunakan baseline mean-reversion.")

if RUN_MODE == "final" and FINAL_MODEL is not None:
    infer_model = FINAL_MODEL
    nm1, nm2    = FINAL_NORM_M1, FINAL_NORM_M2
    mm1, mm2    = FINAL_MEAN_M1, FINAL_MEAN_M2
    cm1, cm2    = FINAL_COEFFS_M1, FINAL_COEFFS_M2
    af_inf, ab_inf = (None, None) if MODEL_NAME == "graph_stid" else (A_FWD_T, A_BWD_T)
elif LAST_INFER_MODEL is not None:
    infer_model = LAST_INFER_MODEL
    nm1, nm2    = LAST_NORM_M1, LAST_NORM_M2
    mm1, mm2    = LAST_MEAN_M1, LAST_MEAN_M2
    cm1, cm2    = LAST_COEFFS_M1, LAST_COEFFS_M2
    af_inf, ab_inf = (None, None) if MODEL_NAME == "graph_stid" else (A_FWD_T, A_BWD_T)
else:
    infer_model = None
    print("Tidak ada model. Menggunakan baseline mean-reversion sebagai fallback.")

if infer_model is not None:
    t0 = time.time()
    PRED_CUBE = inference_test(
        model=infer_model,
        test_hist_np=test_hist, test_regimes_np=test_regimes,
        norm_m1=nm1, norm_m2=nm2,
        mean_m1=mm1, mean_m2=mm2,
        coeffs_m1=cm1, coeffs_m2=cm2,
        az_m1=always_zero_m1, az_m2=always_zero_m2,
        a_fwd=af_inf, a_bwd=ab_inf,
    )
    print(f"Inferensi selesai dalam {time.time()-t0:.1f} dtk | shape={PRED_CUBE.shape}")
else:
    # Fallback baseline mean-reversion
    tr_all_m1 = np.arange(HIST_LEN-1, len(speed_m1)-max(HORIZONS))
    tr_all_m2 = np.arange(HIST_LEN-1, len(speed_m2)-max(HORIZONS))
    nm1 = SpeedNormalizer().fit(speed_m1)
    nm2 = SpeedNormalizer().fit(speed_m2)
    mm1, cm1 = compute_mean_reversion_coeffs(speed_m1, tr_all_m1, stride=30)
    mm2, cm2 = compute_mean_reversion_coeffs(speed_m2, tr_all_m2, stride=30)
    PRED_CUBE = np.zeros((N_TEST, len(HORIZONS), N_ROADS), dtype=np.float32)
    for i in range(N_TEST):
        window = test_hist[i]
        regime = test_regimes[i]
        lm     = mm1  if regime == "m1" else mm2
        coeffs = cm1  if regime == "m1" else cm2
        lr     = window[-1].astype(np.float32)
        for h_idx, h in enumerate(HORIZONS):
            PRED_CUBE[i, h_idx, :] = lr + coeffs[h] * (lm - lr)
    PRED_CUBE = np.clip(PRED_CUBE, CFG["PREDICTION_CLIP"][0], CFG["PREDICTION_CLIP"][1])
    print(f"Fallback baseline: shape={PRED_CUBE.shape}")

assert np.isfinite(PRED_CUBE).all(), "GAGAL: terdapat NaN/Inf pada prediksi!"
print("Prediksi finite dan dalam rentang: LULUS")

In [ ]:
# =============================================================
# CELL 20 - ENSEMBLE OOF PER HORIZON
# =============================================================
# Ensure inference variables exist
try:
    _ = mm1
except NameError:
    tr_all_m1 = np.arange(HIST_LEN-1, len(speed_m1)-max(HORIZONS))
    tr_all_m2 = np.arange(HIST_LEN-1, len(speed_m2)-max(HORIZONS))
    nm1 = SpeedNormalizer().fit(speed_m1)
    nm2 = SpeedNormalizer().fit(speed_m2)
    mm1, cm1 = compute_mean_reversion_coeffs(speed_m1, tr_all_m1, stride=30)
    mm2, cm2 = compute_mean_reversion_coeffs(speed_m2, tr_all_m2, stride=30)


FINAL_PRED_CUBE = PRED_CUBE.copy()

if len(cv_results) >= 2:
    print("=" * 60)
    print("ENSEMBLE OOF PER HORIZON")
    print("=" * 60)

    WEIGHT_GRID = np.arange(0.0, 1.01, 0.1)
    ensemble_weights = {}

    for h_idx, h in enumerate(HORIZONS):
        best_mse, best_w = float("inf"), (1.0, 0.0)
        selection_folds  = [r for r in cv_results if r["fold"] in [0, 1]] or cv_results

        for w_m in WEIGHT_GRID:
            w_b = 1.0 - w_m
            mse_list = []
            for r in selection_folds:
                e_m1 = w_m * r["m1"].get(f"h{h}", 9999) + w_b * r["m1"].get(f"h{h}_baseline", 9999)
                e_m2 = w_m * r["m2"].get(f"h{h}", 9999) + w_b * r["m2"].get(f"h{h}_baseline", 9999)
                mse_list.append(REGIME_WEIGHTS["m1"]*e_m1 + REGIME_WEIGHTS["m2"]*e_m2)
            mean_est = float(np.mean(mse_list))
            if mean_est < best_mse:
                best_mse, best_w = mean_est, (w_m, w_b)

        ensemble_weights[h] = best_w
        print(f"  h{h}: w_model={best_w[0]:.1f}, w_base={best_w[1]:.1f} (est={best_mse:.4f})")

    # Bangun kubus baseline test
    bl_test = np.zeros((N_TEST, len(HORIZONS), N_ROADS), dtype=np.float32)
    for i in range(N_TEST):
        window = test_hist[i]
        regime = test_regimes[i]
        lm     = mm1  if regime == "m1" else mm2
        coeffs = cm1  if regime == "m1" else cm2
        lr     = window[-1].astype(np.float32)
        for h_idx, h in enumerate(HORIZONS):
            bl_test[i, h_idx, :] = lr + coeffs[h] * (lm - lr)

    ens = np.zeros_like(PRED_CUBE)
    for h_idx, h in enumerate(HORIZONS):
        w_m, w_b = ensemble_weights[h]
        ens[:, h_idx, :] = w_m * PRED_CUBE[:, h_idx, :] + w_b * bl_test[:, h_idx, :]
    FINAL_PRED_CUBE = np.clip(ens, CFG["PREDICTION_CLIP"][0], CFG["PREDICTION_CLIP"][1])
    print("Ensemble diterapkan.")
else:
    print(f"{len(cv_results)} fold tersedia. Melewati ensemble (min 2 diperlukan).")

assert np.isfinite(FINAL_PRED_CUBE).all(), "GAGAL: ensemble menghasilkan NaN/Inf!"
print(f"FINAL_PRED_CUBE shape: {FINAL_PRED_CUBE.shape}")

In [ ]:
# =============================================================
# CELL 21 - PEMBUATAN SUBMISSION DAN ASSERTIONS
# =============================================================
# Ensure FINAL_PRED_CUBE exists (jika ensemble dilewati)
try:
    _ = FINAL_PRED_CUBE
except NameError:
    FINAL_PRED_CUBE = PRED_CUBE.copy()


# Verifikasi urutan ID
s_idx = np.repeat(np.arange(N_TEST), len(HORIZONS) * N_ROADS)
h_idx = np.tile(np.repeat([5, 10, 15], N_ROADS), N_TEST)
r_idx = np.tile(np.arange(N_ROADS), N_TEST * len(HORIZONS))
expected_ids = (
    "test_" + pd.Series(s_idx).astype(str).str.zfill(5)
    + "_h"  + pd.Series(h_idx).astype(str)
    + "_r"  + pd.Series(r_idx).astype(str)
)
assert (expected_ids.values == sample_sub["id"].values).all(), \
    "GAGAL: urutan id tidak cocok dengan sample_submission.csv!"
print("Verifikasi urutan ID: LULUS")


def make_submission(
    pred_cube: np.ndarray,
    path: Path,
    clip_range: Tuple[float, float] = (0.0, 160.0),
) -> pd.DataFrame:
    """Membuat submission.csv dari pred_cube (N_TEST, 3, N_ROADS)."""
    assert pred_cube.shape == (N_TEST, len(HORIZONS), N_ROADS), \
        f"Shape salah: {pred_cube.shape}"
    pred_c = np.clip(pred_cube, clip_range[0], clip_range[1])
    assert np.isfinite(pred_c).all(), "GAGAL: terdapat NaN/Inf!"

    sub = sample_sub.copy()
    sub["speed"] = pred_c.reshape(-1)

    assert sub.shape          == (N_TEST * len(HORIZONS) * N_ROADS, 2)
    assert sub.columns.tolist() == ["id", "speed"]
    assert (sub["id"].values  == sample_sub["id"].values).all()
    assert np.isfinite(sub["speed"]).all()
    assert sub["speed"].between(clip_range[0], clip_range[1]).all()

    sub.to_csv(path, index=False)
    return sub


sub_path = WORK_DIR / "submission.csv"
submission = make_submission(FINAL_PRED_CUBE, sub_path, tuple(CFG["PREDICTION_CLIP"]))

print(f"\nSubmission disimpan ke: {sub_path}")
print(f"Jumlah baris: {len(submission):,}")
print(f"Rentang speed: [{submission['speed'].min():.2f}, {submission['speed'].max():.2f}]")
print(submission["speed"].describe().round(3).to_string())
print("\nContoh baris:")
print(submission.head(3).to_string(index=False))
print("...")
print(submission.tail(3).to_string(index=False))
print("\nSeluruh assertions submission: LULUS")

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Ringkasan Hasil <a name="20"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

## Konfigurasi Run

| Parameter | Nilai |
|---|---|
| Model | _(diisi setelah run)_ |
| Mode | _(diisi setelah run)_ |
| Fold | _(diisi setelah run)_ |
| Epoch terbaik | _(diisi setelah run)_ |

## Hasil MSE Validasi

| | h5 | h10 | h15 | Rata-rata |
|---|---|---|---|---|
| Baseline mean-reversion | - | - | - | sekitar 40-42 |
| Model | _(diisi setelah run)_ | _(diisi setelah run)_ | _(diisi setelah run)_ | _(diisi setelah run)_ |

## Status Acceptance Gate

| Syarat | Status |
|---|---|
| MSE < 39.00 | _(diisi setelah run)_ |
| Mengalahkan baseline >= 2/3 fold | _(diisi setelah run)_ |
| MSE h15 tidak lebih buruk | _(diisi setelah run)_ |
| Regime tidak memburuk > 1% | _(diisi setelah run)_ |
| Variasi fold dilaporkan | _(diisi setelah run)_ |

> Ringkasan ini harus diperbarui dari keluaran aktual setelah notebook dieksekusi penuh. Tidak ada hasil yang diklaim sebelum benar-benar dijalankan.

---

## Daftar Pustaka

Shi, H., Wang, P., Liu, H., dan Wang, T. (2022). *Spatial-Temporal Identity: A Simple yet Effective Baseline for Multivariate Time Series Forecasting*. CIKM 2022.

Wu, Z., Pan, S., Long, G., Jiang, J., dan Zhang, C. (2019). *Graph WaveNet for Deep Spatial-Temporal Graph Modeling*. IJCAI 2019.

Harris, C. R. dkk. (2020). *Array programming with NumPy*. Nature, 585, 357-362.

Virtanen, P. dkk. (2020). *SciPy 1.0: Fundamental Algorithms for Scientific Computing in Python*. Nature Methods, 17, 261-272.

Paszke, A. dkk. (2019). *PyTorch: An Imperative Style, High-Performance Deep Learning Library*. NeurIPS 2019.

---

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Terima Kasih! <a name=""></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

**Terima kasih kepada:**

Panitia **Datathon 2026** yang menyediakan *dataset* dan tantangan peramalan lalu lintas yang kaya sinyal.

Tim pengembang *open source*: NumPy, SciPy, PyTorch, dan pandas.